<a href="https://colab.research.google.com/github/jonathanahc/Ejercicios/blob/main/Funciones_y_Programacion_Modular_en_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Funciones y programación modular en C**


## **1. El punto de partida: todo está en `main()`**


Imagina una tienda que vende un producto. Para calcular el pago hace tres operaciones: subtotal, impuesto (16 % en este ejemplo didáctico) y total. Nuestra primera solución reúne todo en `main()`.

**Observa:** ¿qué tendríamos que repetir si calculáramos muchas ventas?

In [1]:
%%writefile venta_01.c
#include <stdio.h>

int main(void)
{
    float precio = 120.0f;
    int cantidad = 3;
    float subtotal = precio * cantidad;
    float impuesto = subtotal * 0.16f;
    float total = subtotal + impuesto;

    printf("Subtotal: %.2f\n", subtotal);
    printf("Impuesto: %.2f\n", impuesto);
    printf("Total: %.2f\n", total);
    return 0;
}

Writing venta_01.c


In [2]:
!gcc -Wall -Wextra -std=c11 venta_01.c -o venta_01 && ./venta_01

Subtotal: 360.00
Impuesto: 57.60
Total: 417.60


### **¿Por qué modularizar?**

El resultado es correcto, pero **la responsabilidad de `main()` es demasiado amplia**: calcula y presenta. Si cambiamos una operación en distintas partes del programa, puede que olvidemos actualizar alguna.

**Programación modular** significa dividir un problema en unidades pequeñas con tareas claras. En C, esas unidades suelen ser **funciones**. En nuestra tienda, una función podría calcular el subtotal, otra el impuesto y otra mostrar el pago.

## **2. Nuestra primera función: extraer una responsabilidad**

Una **función** agrupa instrucciones que realizan una tarea. En el ejemplo siguiente, `mostrarBienvenida()` se encarga de imprimir un mensaje, mientras `main()` controla qué se ejecuta.

`main()` es el punto de entrada del programa; llamar a una función transfiere temporalmente el control hacia ella. `void` indica que esta función **no devuelve un valor**.

In [3]:
%%writefile venta_02.c
#include <stdio.h>

void mostrarBienvenida(void)
{
    printf("Bienvenido al sistema de ventas\n");
}

int main(void)
{
    mostrarBienvenida();
    printf("Continuamos en main()\n");
    return 0;
}

Writing venta_02.c


In [4]:
!gcc -Wall -Wextra -std=c11 venta_02.c -o venta_02 && ./venta_02

Bienvenido al sistema de ventas
Continuamos en main()


## **3. Parámetros y argumentos: la función recibe información**

- **Parámetros**: variables declaradas en la definición de la función (`precio`, `cantidad`).
- **Argumentos**: expresiones o valores enviados cuando se llama (`120.0f`, `3`).

La función siguiente calcula el subtotal y devuelve un número de tipo `float` usando `return`. A diferencia de `void`, **sí produce un valor que podemos guardar o usar en otra operación**.

In [5]:
%%writefile venta_03.c
#include <stdio.h>

float calcularSubtotal(float precio, int cantidad)
{
    return precio * cantidad;
}

int main(void)
{
    float subtotal1 = calcularSubtotal(120.0f, 3);
    float subtotal2 = calcularSubtotal(45.5f, 2);
    printf("Venta 1: %.2f\n", subtotal1);
    printf("Venta 2: %.2f\n", );
    return 0;
}

Writing venta_03.c


In [6]:
!gcc -Wall -Wextra -std=c11 venta_03.c -o venta_03 && ./venta_03

Venta 1: 360.00
Venta 2: 91.00


## **4. Prototipos: anunciar las funciones antes de usarlas**

Hasta ahora definimos cada función por encima de `main()`. También podemos escribir primero su **prototipo**, usarla en `main()` y dejar la definición completa más abajo.

Un prototipo indica el **tipo de retorno, nombre y tipos de parámetros**. Termina en `;` porque no contiene cuerpo. Así, el compilador conoce la función al encontrar su llamada.

In [7]:
%%writefile venta_04.c
#include <stdio.h>

// Prototipos
float calcularSubtotal(float precio, int cantidad);
float calcularImpuesto(float subtotal);
void mostrarResumen(float subtotal, float impuesto, float total);

int main(void)
{
    float precio = 120.0f;
    int cantidad = 3;
    float subtotal = calcularSubtotal(precio, cantidad);
    float impuesto = calcularImpuesto(subtotal);
    float total = subtotal + impuesto;
    mostrarResumen(subtotal, impuesto, total);
    return 0;
}

// Definiciones
float calcularSubtotal(float precio, int cantidad)
{
    return precio * cantidad;
}

float calcularImpuesto(float subtotal)
{
    return subtotal * 0.16f;
}

void mostrarResumen(float subtotal, float impuesto, float total)
{
    printf("Subtotal: %.2f\n", subtotal);
    printf("Impuesto: %.2f\n", impuesto);
    printf("Total: %.2f\n", total);
}

Writing venta_04.c


In [8]:
!gcc -Wall -Wextra -std=c11 venta_04.c -o venta_04 && ./venta_04

Subtotal: 360.00
Impuesto: 57.60
Total: 417.60


### **¿Qué cambió con respecto a la primera versión?**

Los cálculos ya no están concentrados en `main()`. Ahora `main()` **coordina**: llama, recibe resultados y pide que se muestren.

| Concepto | Dónde aparece | Función |
|---|---|---|
| Prototipo | `float calcularSubtotal(...);` | Anuncia una función |
| Parámetros | `float precio, int cantidad` | Reciben los datos |
| Argumentos | `calcularSubtotal(precio, cantidad)` | Aportan valores |
| `return` | `return precio * cantidad;` | Devuelve resultado |
| `void` | `void mostrarResumen(...)` | No devuelve resultado |
| `main()` | `int main(void)` | Organiza el flujo |

## **5. Variables locales: mismo nombre, espacios distintos**

Una variable declarada dentro de un bloque posee **alcance local**. Que dos funciones usen el mismo nombre **no significa** que compartan la misma variable.

En este ejemplo `cantidad` dentro de `cambiarCantidad()` es un parámetro local: modificarlo **no cambia** la variable `cantidad` de `main()`.

In [9]:
%%writefile venta_05.c
#include <stdio.h>

void cambiarCantidad(int cantidad)
{
    cantidad = 99;
    printf("Dentro de la funcion: %d\n", cantidad);
}

int main(void)
{
    int cantidad = 3;
    cambiarCantidad(cantidad);
    printf("En main(): %d\n", cantidad);
    return 0;
}

Writing venta_05.c


In [10]:
!gcc -Wall -Wextra -std=c11 venta_05.c -o venta_05 && ./venta_05

Dentro de la funcion: 99
En main(): 3


## **6. Variables globales: datos compartidos, pero con cuidado**

Una variable declarada **fuera de las funciones**, a nivel de archivo, puede ser utilizada por funciones que la tengan visible. Es útil en ciertos casos, pero genera dependencias: cualquier función con acceso puede cambiar su valor.

Aquí registramos cuántas ventas se procesaron.

In [11]:
%%writefile venta_06.c
#include <stdio.h>

int ventasProcesadas = 0; // Variable global

void registrarVenta(void)
{
    ventasProcesadas++;
}

int main(void)
{
    registrarVenta();
    registrarVenta();
    printf("Ventas procesadas: %d\n", ventasProcesadas);
    return 0;
}

Writing venta_06.c


In [12]:
!gcc -Wall -Wextra -std=c11 venta_06.c -o venta_06 && ./venta_06

Ventas procesadas: 2


## **7. Paso por valor y modificación mediante apuntadores**

En **C todos los argumentos se pasan por valor**. Cuando el argumento es una dirección de memoria, lo que se copia es **esa dirección**; a través de ella, una función puede cambiar el objeto original. A este patrón suele llamársele de manera informal *paso por referencia*, aunque C no tiene parámetros de referencia como característica propia.

Primero comprobamos que modificar un número recibido por valor no cambia la variable de origen.

In [13]:
%%writefile venta_07.c
#include <stdio.h>

void agregarUnidadesPorValor(int cantidad)
{
    cantidad = cantidad + 2;
    printf("En funcion (copia): %d\n", cantidad);
}

int main(void)
{
    int cantidad = 3;
    agregarUnidadesPorValor(cantidad);
    printf("En main (original): %d\n", cantidad);
    return 0;
}

Writing venta_07.c


In [14]:
!gcc -Wall -Wextra -std=c11 venta_07.c -o venta_07 && ./venta_07

En funcion (copia): 5
En main (original): 3


### **Ahora modificamos el dato original**

- `&cantidad` obtiene la **dirección** de la variable.
- `int *cantidad` declara un parámetro que recibe una **dirección de `int`**.
- `*cantidad` accede al **valor almacenado** en esa dirección.

Es el mismo principio de los ejercicios de apuntadores, pero ahora aplicado a una función.

In [15]:
%%writefile venta_08.c
#include <stdio.h>

void agregarUnidades(int *cantidad)
{
    *cantidad = *cantidad + 2;
}

int main(void)
{
    int cantidad = 3;
    printf("Antes: %d\n", cantidad);
    agregarUnidades(&cantidad);
    printf("Despues: %d\n", cantidad);
    return 0;
}

Writing venta_08.c


In [16]:
!gcc -Wall -Wextra -std=c11 venta_08.c -o venta_08 && ./venta_08

Antes: 3
Despues: 5


| Pregunta | Paso por valor de un `int` | Pasar la dirección de un `int` |
|---|---|---|
| Llamada | `funcion(cantidad)` | `funcion(&cantidad)` |
| Parámetro | `int cantidad` | `int *cantidad` |
| Qué se copia | El número | La dirección |
| Puede modificar el original | No, solo modifica su copia | Sí, si desreferencia la dirección |

## **8. Pila de llamadas: ¿quién regresa a dónde?**

Una función puede llamar a otra. El sistema conserva la información necesaria para regresar a cada llamada; conceptualmente esto se organiza en una **pila de llamadas**.

Sigamos el flujo del sistema: `main()` llama a `procesarVenta()`, que llama a `calcularSubtotal()`. Cuando la función interna termina, el control vuelve a quien la llamó.

```text
Al entrar a calcularSubtotal():     Al finalizar calcularSubtotal():
┌────────────────────────┐           ┌────────────────────────┐
│ calcularSubtotal()     │           │ procesarVenta()        │
├────────────────────────┤           ├────────────────────────┤
│ procesarVenta()        │           │ main()                 │
├────────────────────────┤           └────────────────────────┘
│ main()                 │
└────────────────────────┘
```

Esta es una representación simplificada: el detalle exacto de la memoria depende de la implementación.

In [ ]:
%%writefile venta_09.c
#include <stdio.h>

float calcularSubtotal(float precio, int cantidad)
{
    puts("3. Entramos a calcularSubtotal()");
    return precio * cantidad;
}

void procesarVenta(void)
{
    puts("2. Entramos a procesarVenta()");
    float subtotal = calcularSubtotal(100.0f, 2);
    printf("4. Regresamos a procesarVenta(): %.2f\n", subtotal);
}

int main(void)
{
    puts("1. Inicia main()");
    procesarVenta();
    puts("5. Regresamos a main()");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -std=c11 venta_09.c -o venta_09 && ./venta_09

**Fíjate en el orden `1 → 2 → 3 → 4 → 5`.** Las funciones regresan en orden inverso al de las llamadas: primero termina la más interna.

## **9. Proyecto integrador guiado: venta con inventario**

El programa final debe:
1. Tener **prototipos** antes de `main()`.
2. Calcular subtotal, impuesto y total con funciones que **devuelven resultados**.
3. Imprimir el resumen con una función `void`.
4. Descontar unidades del inventario **mediante un apuntador**.
5. Dejar a `main()` la coordinación general.

**Antes de ejecutarlo, predice:** ¿cuál será el inventario final si empezamos con 10 unidades y vendemos 3?

In [17]:
%%writefile venta_final.c
#include <stdio.h>

float calcularSubtotal(float precio, int cantidad);
float calcularImpuesto(float subtotal);
void descontarInventario(int *inventario, int cantidad);
void mostrarResumen(float subtotal, float impuesto, float total, int inventario);

int main(void)
{
    float precio = 120.0f;
    int cantidad = 3;
    int inventario = 10;

    if (cantidad <= 0 || cantidad > inventario)
    {
        puts("Cantidad no valida o inventario insuficiente.");
        return 1;
    }

    float subtotal = calcularSubtotal(precio, cantidad);
    float impuesto = calcularImpuesto(subtotal);
    float total = subtotal + impuesto;

    descontarInventario(&inventario, cantidad);
    mostrarResumen(subtotal, impuesto, total, inventario);
    return 0;
}

float calcularSubtotal(float precio, int cantidad)
{
    return precio * cantidad;
}

float calcularImpuesto(float subtotal)
{
    return subtotal * 0.16f;
}

void descontarInventario(int *inventario, int cantidad)
{
    *inventario -= cantidad;
}

void mostrarResumen(float subtotal, float impuesto, float total, int inventario)
{
    printf("Subtotal: %.2f\n", subtotal);
    printf("Impuesto: %.2f\n", impuesto);
    printf("Total: %.2f\n", total);
    printf("Inventario restante: %d\n", inventario);
}

Writing venta_final.c


In [18]:
!gcc -Wall -Wextra -std=c11 venta_final.c -o venta_final && ./venta_final

Subtotal: 360.00
Impuesto: 57.60
Total: 417.60
Inventario restante: 7
